# 05 - Phân tích

## 0.Chuẩn bị

In [12]:
import re, sys, json, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from IPython.display import display
from collections import Counter

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)

from src.config import LABELS, DATA_DIR, MODELS_DIR, RESULTS_METRICS, RESULTS_PREDICTIONS, TRANSFORMERS, text_col
from preprocess import TEENCODE, EMOJI_RE     

MODELS = [m for m in ("baseline", "phobert", "visobert") if (RESULTS_PREDICTIONS / f"{m}_test.csv").exists()]
CLASSES = [LABELS[i] for i in sorted(LABELS)]
assert MODELS, "Chưa có file dự đoán nào trong results/predictions/"

train, val, test = (pd.read_csv(DATA_DIR / f"{s}.csv") for s in ("train", "val", "test"))

def load_pred(m, split, ref):
    '''đọc file dự đoán và kiểm tra khớp từng dòng với test.csv / val.csv'''
    p = pd.read_csv(RESULTS_PREDICTIONS / f"{m}_{split}.csv")
    assert len(p) == len(ref), f"{m}/{split}: số dòng dự đoán ({len(p)}) khác {split}.csv ({len(ref)}) - có thể đã chạy với --limit"
    assert (p["true_id"].values == ref["label_id"].values).all(), f"{m}/{split}: nhãn thật không khớp"
    assert (p["text"].fillna("").values == ref[text_col(m)].fillna("").values).all(), f"{m}/{split}: văn bản không khớp text_{m}"
    if "id" not in p.columns: p.insert(0, "id", ref["id"].values)
    if "free_text" not in p.columns: p.insert(1, "free_text", ref["free_text"].values)
    assert (p["id"].values == ref["id"].values).all(), f"{m}/{split}: id không khớp"
    p["free_text"] = p["free_text"].fillna("")
    return p

P = {m: load_pred(m, "test", test) for m in MODELS}  
V = {m: load_pred(m, "val", val) for m in MODELS}        
VT = {m: pd.concat([V[m].assign(split="val"), P[m].assign(split="test")], ignore_index=True) for m in MODELS}
y = test["label_id"].values
print("Mô hình:", MODELS, "| test:", len(test), "| val:", len(val))


PBT = f"p_{LABELS[0]}"                                   # xác suất lớp BÌNH THƯỜNG
def toxic(p):  return (p["true_id"].values != 0)         # nhãn thật: độc hại (XÚC PHẠM + THÙ GHÉT)
def flagged(p): return (p["pred_id"].values != 0)        # mô hình đoán: độc hại
def score(p):  return 1.0 - p[PBT].values                # điểm độc hại = 1 - P(BÌNH THƯỜNG)

def pct(k, n):
    return f"{k / n * 100:.1f}%" if n else "-"

def pct1(x):                       # x là tỉ lệ 0-1
    return f"{x * 100:.1f}%"

#từ điển thô tục
# VULGAR: từ tục/xúc phạm trực tiếp (hay bị che để né bộ lọc)
# EXPLETIVE: từ cảm thán viết tắt (vl, vcl, cmn...) thường chỉ nhấn mạnh
VULGAR = ["địt", "đụ", "lồn", "cặc", "buồi", "đéo", "cứt", "đĩ", "ngu", "chó", "điên", "khùng", "đần", "dốt",
          "mất dạy", "khốn nạn", "súc vật", "súc sinh", "óc chó", "vô học"]
EXPLETIVE = ["đm", "dm", "dmm", "đmm", "đcm", "dcm", "đkm", "dkm", "đcmm", "vcl", "vkl", "vl", "cmn", "cmnr", "clgt", "clm", "cc", "cl", "đù", "lol", "loz", "vãi"]
def term_re(terms):
    return re.compile("|".join(rf"(?<!\w){re.escape(t)}(?!\w)" for t in sorted(terms, key=len, reverse=True)), re.I)
VULGAR_RE, EXPLETIVE_RE = term_re(VULGAR), term_re(EXPLETIVE)
def has_term(series, regex):
    return series.fillna("").str.lower().map(lambda s: bool(regex.search(s))).values

Mô hình: ['baseline', 'phobert', 'visobert'] | test: 2985 | val: 3003


## 1. FPR và FNR

- FP là câu BÌNH THƯỜNG bị đoán XÚC PHẠM/THÙ GHÉT
- FN là câu độc hại bị đoán BÌNH THƯỜNG

In [ ]:
# độc hại = XÚC PHẠM + THÙ GHÉT, so với BÌNH THƯỜNG
rows = []
for m in MODELS:
    p = P[m]; t, f = toxic(p), flagged(p)
    n_norm, n_tox = int((~t).sum()), int(t.sum())
    fp, fn, tp = int((f & ~t).sum()), int((~f & t).sum()), int((f & t).sum())
    rows.append(
        {
            "mô hình": m, 
            "FP (báo động giả)": fp, 
            "FPR": pct(fp, n_norm), 
            "FN (bỏ sót)": fn, 
            "FNR": pct(fn, n_tox),
            "precision (độc hại)": pct(tp, tp + fp), 
            "recall (độc hại)": pct(tp, tp + fn), 
            "F1 (độc hại)": pct(2 * tp, 2 * tp + fp + fn)
        }
    )
display(pd.DataFrame(rows).set_index("mô hình"))
print(f"{int((~toxic(P[MODELS[0]])).sum())} câu BÌNH THƯỜNG (cho FPR), {int(toxic(P[MODELS[0]]).sum())} câu độc hại (cho FNR)")

# FP đi về lớp nào, FN đến từ lớp nào
rows = []
for m in MODELS:
    p = P[m]
    fn_xp, fn_tg = int(((p.true_id == 1) & (p.pred_id == 0)).sum()), int(((p.true_id == 2) & (p.pred_id == 0)).sum())
    rows.append(
        {
            "mô hình": m,
            "FP → XÚC PHẠM": int(((p.true_id == 0) & (p.pred_id == 1)).sum()), 
            "FP → THÙ GHÉT": int(((p.true_id == 0) & (p.pred_id == 2)).sum()),
            "FN từ XÚC PHẠM": fn_xp, 
            "FN từ THÙ GHÉT": fn_tg,
            "FNR XÚC PHẠM": pct(fn_xp, int((p.true_id == 1).sum())), 
            "FNR THÙ GHÉT": pct(fn_tg, int((p.true_id == 2).sum()))
        }
    )
display(pd.DataFrame(rows).set_index("mô hình"))

,FP (báo động giả),FPR,FN (bỏ sót),FNR,precision (độc hại),recall (độc hại),F1 (độc hại)
mô hình,,,,,,,
baseline,123,5.0%,162,30.7%,74.8%,69.3%,71.9%
phobert,209,8.5%,113,21.4%,66.5%,78.6%,72.0%
visobert,166,6.8%,153,29.0%,69.3%,71.0%,70.1%


2458 câu BÌNH THƯỜNG (cho FPR), 527 câu độc hại (cho FNR)


,FP → XÚC PHẠM,FP → THÙ GHÉT,FN từ XÚC PHẠM,FN từ THÙ GHÉT,FNR XÚC PHẠM,FNR THÙ GHÉT
mô hình,,,,,,
baseline,52,71,74,88,36.8%,27.0%
phobert,109,100,56,57,27.9%,17.5%
visobert,84,82,63,90,31.3%,27.6%


F1 của độc hại gần như bằng nhau (71,9 - 72 - 70,1):
- baseline ít báo động giả nhất (FPR 5%) nhưng bỏ sót nhiều nhất (FNR 30,7%)
- phobert ngược lại báo động giả ít nhất nhưng bỏ sót nhiều nhất
- visobert ở giữa
Báo động giả của phobert chia khá đều cho XÚC PHẠM và THÙ GHÉT, baseline chủ yếu ở THÙ GHÉT

XÚC PHẠM bị bot sót nhiều hơn THÙ GHÉT ở cả 3 mô hình -> lớp có ranh giới không rõ ràng

## 2. Mất cân bằng dữ liệu

BÌNH THƯỜNG 82%, THÙ GHÉT 11%, XÚC PHẠM 7%

### 2.1. Accuracy

In [ ]:
always_bt = np.zeros_like(y)
rows = [{"bộ phân loại": "luôn đoán BÌNH THƯỜNG (không học gì)", "accuracy": pct1((always_bt == y).mean()),
         "macro-F1": pct1(f1_score(y, always_bt, average="macro", zero_division=0)), "FNR": "100.0%"}]
for m in MODELS:
    t, f = toxic(P[m]), flagged(P[m])
    rows.append(
        {
            "bộ phân loại": m, 
            "accuracy": pct1(P[m].correct.mean()), 
            "macro-F1": pct1(f1_score(y, P[m].pred_id, average="macro")), 
            "FNR": pct1((~f[t]).mean())
        }
    )
display(pd.DataFrame(rows).set_index("bộ phân loại"))

,accuracy,macro-F1,FNR
bộ phân loại,,,
luôn đoán BÌNH THƯỜNG (không học gì),82.3%,30.1%,100.0%
baseline,87.8%,67.0%,30.7%
phobert,86.6%,69.1%,21.4%
visobert,86.9%,67.4%,29.0%


### 2.2. Tỉ lệ nhãn thật và nhãn dự đoán

In [6]:
def cw_setting(m):
    if m == "baseline": 
        return "class_weight='balanced'"
    
    c = TRANSFORMERS[m]
    if c["class_weight"]:
        return f"loss có trọng số, power={c['class_weight_power']}"
    else:
        return "không dùng trọng số"
    
rows = []
true_share = pd.Series(y).map(LABELS).value_counts(normalize=True).reindex(CLASSES) * 100
rows.append(
    {
        "": "nhãn thật (test)", 
        **{c: f"{true_share[c]:.1f}%" for c in CLASSES}, 
        "xử lý mất cân bằng": "-"
    }
)
for m in MODELS:
    ps = P[m].pred_label.value_counts(normalize=True).reindex(CLASSES).fillna(0) * 100
    rows.append(
        {
            "": f"đoán bởi {m}", 
            **{c: f"{ps[c]:.1f}% ({ps[c] / true_share[c]:.2f}x)" for c in CLASSES}, 
            "xử lý mất cân bằng": cw_setting(m)
        }
    )
display(pd.DataFrame(rows).set_index(""))

,BÌNH THƯỜNG,XÚC PHẠM,THÙ GHÉT,xử lý mất cân bằng
,,,,
nhãn thật (test),82.3%,6.7%,10.9%,-
đoán bởi baseline,83.7% (1.02x),5.5% (0.82x),10.8% (0.99x),class_weight='balanced'
đoán bởi phobert,79.1% (0.96x),9.2% (1.36x),11.7% (1.07x),"loss có trọng số, power=0.5"
đoán bởi visobert,81.9% (0.99x),6.9% (1.02x),11.2% (1.03x),"loss có trọng số, power=1.0"


### 2.3. cả 3 mô hình cùng sai lớp XÚC PHẠM

In [7]:
wrong_conditions = [~P[m].correct.values for m in MODELS]
wrong_all = np.logical_and.reduce(wrong_conditions)

t_true = P[MODELS[0]].true_label

out = pd.DataFrame({
    "cả 3 mô hình cùng sai": pd.Series(wrong_all).groupby(t_true.values).sum(), 
    "tổng câu": t_true.value_counts()
}).reindex(CLASSES)

out["% cả 3 cùng sai"] = [
    pct(failed_count, total_count) 
    for failed_count, total_count in zip(out.iloc[:, 0], out["tổng câu"])
]

display(out)

,cả 3 mô hình cùng sai,tổng câu,% cả 3 cùng sai
BÌNH THƯỜNG,59,2458,2.4%
XÚC PHẠM,60,201,29.9%
THÙ GHÉT,58,326,17.8%


**Nhận xét mất cân bằng dữ liệu**

- bộ phân loại "luôn đoán BÌNH THƯỜNG" đạt accuracy 82,3% nhưng macro-f1 chỉ 30,1%. nên accuracy không dùng để so sánh dữ liệu mất cân bằng, macro-f1 và FNR mới phản ánh đúng
- cả 3 mô hình có tỉ lệ dự đoán nhãn gần với tỉ lệ thật trừ XÚC PHẠM: phobert đoán thừa rất nhiều, do việc chuẩn hóa teencode ` normalization.json `. mô hình này học các từ này sau tiền xử lý cho vào huấn luyện. ở cuối 03_phobert.ipynb có bảng cho thấy câu BÌNH THƯỜNG chứa `m` bị phobert báo động giả 15,5% (baseline 9,9% và visobert 5,6%), chứa `t` 11,4% (1,8% và 7%)
- XÚC PHẠM là lớp bị đoán sai nhiều nhất ở cả 3 lớp: 29,9% bị đoán sai, so với 17,8% của THÙ GHÉT và 2,4% của BÌNH THƯỜNG. lớp chỉ có 201 câu trong test và f1 chỉ 43,7 - 48,8% ở cả 3 mô hình. do nhãn mơ hồ, nhiễu, câu cần hiểu ngữ cảnh cả câu

## 3. Bias với nhóm xã hội

In [8]:
GROUPS = {
    "chính trị - Đảng/nhà nước": ["đảng", "cộng sản", "nhà nước", "chính quyền", "công an", "cán bộ", "lãnh đạo", "chính phủ", "quan chức"],
    "chính trị - nhãn đối lập": ["phản động", "dlv", "dân chủ", "bò đỏ", "cờ vàng", "ba que"],
    "tôn giáo": ["công giáo", "thiên chúa", "phật giáo", "hồi giáo", "tin lành", "phật", "chúa", "nhà thờ", "thầy chùa"],
    "vùng miền": ["bắc kỳ", "nam kỳ", "miền bắc", "miền nam", "miền trung", "hà nội", "sài gòn", "tp hcm", "huế", "nghệ an", "hải phòng"],
    "quốc tịch - Trung Quốc": ["trung quốc", "tq", "khựa", "trung cộng"],
    "quốc tịch - nước khác": ["hàn quốc", "nhật bản", "nhật", "campuchia", "thái lan", "mỹ", "lào", "indonesia", "indo", "ấn độ", "đài loan", "việt kiều"],
    "giới": ["phụ nữ", "đàn bà", "đàn ông", "con gái", "con trai", "chị em"],
    "xu hướng tính dục": ["gay", "đồng tính", "lgbt", "les", "bê đê", "pê đê", "chuyển giới"],
    "nghề nghiệp / tầng lớp": ["bộ đội", "giáo viên", "bác sĩ", "nhà báo", "nghệ sĩ", "ca sĩ", "sinh viên", "nông dân", "người nghèo", "dân nghèo", "shipper", "tài xế"],
}

tr_text = train["free_text"].fillna("")
rows = []

for g, terms in GROUPS.items():
    rg = term_re(terms)
    in_train = has_term(tr_text, rg)
    row = {
        "nhóm": g, 
        "n train": int(in_train.sum()), 
        "% độc hại (train)": pct1((train.label_id.values[in_train] != 0).mean())
    }
    
    for m in MODELS:
        p = VT[m]
        ms = has_term(p.free_text, rg)
        normal = ms & (p.true_id.values == 0)
        
        row["n câu BT"] = int(normal.sum())
        row[f"FPR {m}"] = pct(int((flagged(p) & normal).sum()), int(normal.sum()))
    
    rows.append(row)
first_model = MODELS[0]
total_normal_count = int((VT[first_model].true_id.values == 0).sum())

overall = {
    "nhóm": "(FPR chung, mọi câu BT)", 
    "n train": "", 
    "% độc hại (train)": "", 
    "n câu BT": total_normal_count
}

for m in MODELS:
    p = VT[m] 
    normal_all = (p.true_id.values == 0)
    overall[f"FPR {m}"] = pct(
        int((flagged(p) & normal_all).sum()), 
        int(normal_all.sum())
    )
    
display(pd.DataFrame(rows + [overall]).set_index("nhóm"))

,n train,% độc hại (train),n câu BT,FPR baseline,FPR phobert,FPR visobert
nhóm,,,,,,
chính trị - Đảng/nhà nước,400,51.7%,28,39.3%,53.6%,17.9%
chính trị - nhãn đối lập,153,66.0%,17,41.2%,64.7%,52.9%
tôn giáo,51,21.6%,14,7.1%,0.0%,0.0%
vùng miền,123,31.7%,14,14.3%,21.4%,7.1%
quốc tịch - Trung Quốc,110,47.3%,9,11.1%,22.2%,11.1%
quốc tịch - nước khác,499,26.3%,96,4.2%,9.4%,4.2%
giới,184,19.0%,30,3.3%,10.0%,13.3%
xu hướng tính dục,81,29.6%,13,15.4%,7.7%,7.7%
nghề nghiệp / tầng lớp,143,25.2%,22,13.6%,13.6%,18.2%


**Nhận xét bias nhóm xã hội**
- lệch rõ nhất là chủ đề chính trị: câu bình thường nhắc đến đảng/nhà nước, công an, cán bộ... bị báo động giả 39,3% baseline, 53,6% phobert, 17,9% visobert. nhãn đối lập (phản động, dân chủ...) 41,2% 64,7% 52,9%. trong khi mức chung chỉ 4,9% 8,4% 6,8%. do mô hình này học đặc điểm trên train, mà nhãn độc hại chứa những từ này lên tới 52 - 66% -> dễ đoán sai
- các nhóm còn lại (tôn giáo, vùng miền, quốc tịch...) chi có 9 - 30 câu bình thường, nên nhóm chưa thể kết luận

## 4. Viết tắt nói lái chèn ký tự

In [9]:
VI_CHARS = re.compile("[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", re.I)
MASK_RE = re.compile(r"(?<=\w)[\*@#\$](?=\w)")                         # đ*t, l*n, c**
SPLIT_RE = re.compile(r"(?<!\w)(?:\w[\.\-_ ]){2,}\w(?!\w)")              # l.o.n, đ-ị-t, l o n  (>= 3 chữ tách nhau)
LEET_RE = re.compile(r"\b[a-zA-Z]+[0-9][a-zA-Z0-9]*\b")                 # d4ng, y3u, l0n
ZW_RE = re.compile("[\u200b\u200c\u200d\u2060\ufeff]")

def fancy_unicode(s):   
    return any(0x1D400 <= ord(c) <= 0x1D7FF or 0x1F100 <= ord(c) <= 0x1F1FF or 0xFF21 <= ord(c) <= 0xFF5A for c in s)

def is_caps(s):
    letters = [c for c in s if c.isalpha()]
    return len(letters) >= 8 and sum(c.isupper() for c in letters) / len(letters) > 0.7

base_text = VT[MODELS[0]].free_text
tok_lower = base_text.str.lower().str.replace(r"[^\w\s]", " ", regex=True).str.split()
TE = {k for k in TEENCODE if len(k) >= 2}

FLAGS = {
    "không dấu": ((~base_text.str.contains(VI_CHARS)) & (base_text.str.len() >= 12) & base_text.str.contains(r"[a-zA-Z]{3}")).values,
    "kéo dài chữ (>=3 lần)": base_text.str.contains(r"([^\W\d_])\1{2,}").values,
    "VIẾT HOA TOÀN BỘ": base_text.map(is_caps).values,
    "teencode / viết tắt thường": tok_lower.map(lambda t: any(w in TE for w in t)).values,
    "viết tắt chửi thề (dm, vcl, cc...)": has_term(base_text, EXPLETIVE_RE),
    "có emoji": base_text.str.contains(EMOJI_RE).values,
    "cố ý che/chèn ký tự (*, chấm, leet, zero-width, chữ lạ)": (
        base_text.str.contains(MASK_RE) | 
        base_text.str.contains(SPLIT_RE) | 
        base_text.str.contains(LEET_RE) | 
        base_text.str.contains(ZW_RE) | 
        base_text.map(fancy_unicode)
    ).values,
}

rows = []

for name, f in FLAGS.items():
    row = {
        "đặc điểm": name, 
        "n": int(f.sum()), 
        "% độc hại thật": pct1(toxic(VT[MODELS[0]])[f].mean()) if f.sum() else "-"
    }
    
    for m in MODELS:
        p = VT[m]
        t = toxic(p)     
        fl = flagged(p)  
        
        row[f"{m} FNR"] = pct1((~fl[f & t]).mean()) if (f & t).sum() else "-"

        row[f"{m} FPR"] = pct1(fl[f & ~t].mean()) if (f & ~t).sum() else "-"
        
    rows.append(row)

overall = {
    "đặc điểm": "(toàn bộ val+test)", 
    "n": len(base_text), 
    "% độc hại thật": pct1(toxic(VT[MODELS[0]]).mean())
}

for m in MODELS:
    p = VT[m]
    t = toxic(p)
    fl = flagged(p)
    overall[f"{m} FNR"] = pct1((~fl[t]).mean())
    overall[f"{m} FPR"] = pct1(fl[~t].mean())


df_flags = pd.DataFrame(rows + [overall]).set_index("đặc điểm")
display(df_flags)


n_obf = int(FLAGS["cố ý che/chèn ký tự (*, chấm, leet, zero-width, chữ lạ)"].sum())

C:\Users\this pc\AppData\Local\Temp\ipykernel_7692\519266384.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "kéo dài chữ (>=3 lần)": base_text.str.contains(r"([^\W\d_])\1{2,}").values,


,n,% độc hại thật,baseline FNR,baseline FPR,phobert FNR,phobert FPR,visobert FNR,visobert FPR
đặc điểm,,,,,,,,
không dấu,239,21.8%,28.8%,10.2%,26.9%,14.4%,23.1%,14.4%
kéo dài chữ (>=3 lần),186,11.8%,22.7%,2.4%,22.7%,6.7%,31.8%,4.9%
VIẾT HOA TOÀN BỘ,26,15.4%,75.0%,9.1%,25.0%,36.4%,50.0%,27.3%
teencode / viết tắt thường,1123,24.7%,21.7%,8.0%,13.4%,13.6%,22.7%,10.3%
"viết tắt chửi thề (dm, vcl, cc...)",289,61.6%,12.4%,29.7%,6.2%,45.0%,11.2%,38.7%
có emoji,884,9.4%,43.4%,4.4%,32.5%,6.7%,38.6%,6.1%
"cố ý che/chèn ký tự (*, chấm, leet, zero-width, chữ lạ)",57,15.8%,22.2%,0.0%,11.1%,6.2%,22.2%,6.2%
(toàn bộ val+test),5988,17.6%,30.0%,4.9%,20.7%,8.4%,27.6%,6.8%


**Nhận xét viết tắt, nói lái, chèn lý tự**
- Che/chèn lý tự và viết hoa toàn bộ rất hiếm trong tập dữ liệu nên nhóm không thể kết luận
- viết tắt chửi thể: có báo động giả lớn. Mô hình coi sự xuất hiện của từ viết tắt là độc hại, bất kể từ đó chỉ mang ý nhấn mạnh, đùa
- emoji: có FNR cao nhất, ở phần tiền xử lý emoji nhóm không thể chuẩn hóa hết được nên những emoji không có sẽ bị bỏ sót so với tập dữ liệu ban đầu
- không dấu: FPR tăng gần gấp đôi (10 - 14% so với 5 -8%), FNR không lệch nhiều. mô hình dễ nhầm chữ vô hại không dấu thành chữ tục

## 5. False positive với câu trung tính
vì dữ liệu quá lớn nên nhóm chỉ có thể liệt kê được 1 số trường hợp

In [13]:
FP_ANNOT = {
    # từ cảm thán / thô tục dùng để đùa hoặc nhấn mạnh, không nhắm vào ai
    7998: "tục dùng nhấn mạnh/đùa", 3964: "tục dùng nhấn mạnh/đùa", 17057: "tục dùng nhấn mạnh/đùa", 4205: "tục dùng nhấn mạnh/đùa", 13910: "tục dùng nhấn mạnh/đùa",
    31007: "tục dùng nhấn mạnh/đùa", 12655: "tục dùng nhấn mạnh/đùa", 2657: "tục dùng nhấn mạnh/đùa", 9401: "tục dùng nhấn mạnh/đùa", 13773: "tục dùng nhấn mạnh/đùa",
    # nói về chính trị / nhà nước, không công kích cá nhân hay nhóm
    6086: "chủ đề chính trị / nhà nước", 26013: "chủ đề chính trị / nhà nước", 1405: "chủ đề chính trị / nhà nước", 19171: "chủ đề chính trị / nhà nước", 10614: "chủ đề chính trị / nhà nước",
    # không dấu / viết tắt làm chữ vô hại giống chữ tục
    27439: "không dấu gây nhầm từ", 33773: "không dấu gây nhầm từ", 22495: "không dấu gây nhầm từ",
    # nội dung có vẻ công kích nhưng nhãn gốc là BÌNH THƯỜNG -> nghi nhiễu nhãn
    24790: "nghi nhiễu nhãn", 13571: "nghi nhiễu nhãn", 25366: "nghi nhiễu nhãn", 21350: "nghi nhiễu nhãn", 25287: "nghi nhiễu nhãn", 22622: "nghi nhiễu nhãn", 5579: "nghi nhiễu nhãn",
}

In [14]:
# so sánh đặc điểm hình thức giữa câu báo động giả và câu BÌNH THƯỜNG đúng
rows = []

for m in MODELS: 
    p = VT[m]
    t = toxic(p)    
    fl = flagged(p)  
    
    fp = fl & ~t     
    ok = ~fl & ~t    
    txt = p.free_text
    
    feats = {
        "có từ tục (VULGAR)": has_term(txt, VULGAR_RE), 
        "có từ cảm thán viết tắt (vl, vcl, cc...)": has_term(txt, EXPLETIVE_RE),
        "có emoji": txt.str.contains(EMOJI_RE).values, 
        "ngắn (<= 5 từ)": (txt.str.split().str.len() <= 5).values,
        "không có từ tục lẫn cảm thán": ~has_term(txt, VULGAR_RE) & ~has_term(txt, EXPLETIVE_RE)
    }
    
    # % xuất hiện của từng đặc điểm trong nhóm FP và nhóm OK
    for k, v in feats.items():
        rows.append({
            "mô hình": m, 
            "đặc điểm": k, 
            "% trong FP": pct1(v[fp].mean()), 
            "% trong BT đoán đúng": pct1(v[ok].mean()), 
            "n FP": int(fp.sum())
        })

tab = pd.DataFrame(rows)

display(tab.pivot(index="đặc điểm", columns="mô hình", values="% trong FP")[MODELS].rename_axis("% FP có đặc điểm"))
display(tab.pivot(index="đặc điểm", columns="mô hình", values="% trong BT đoán đúng")[MODELS].rename_axis("% BT-đúng có đặc điểm (mốc so sánh)"))

mô hình,baseline,phobert,visobert
% FP có đặc điểm,,,
có emoji,14.5%,13.0%,14.6%
"có từ cảm thán viết tắt (vl, vcl, cc...)",13.7%,12.1%,12.8%
có từ tục (VULGAR),10.4%,9.7%,9.6%
không có từ tục lẫn cảm thán,76.3%,78.5%,77.9%
ngắn (<= 5 từ),19.9%,18.4%,22.7%


mô hình,baseline,phobert,visobert
% BT-đúng có đặc điểm (mốc so sánh),,,
có emoji,16.3%,16.5%,16.4%
"có từ cảm thán viết tắt (vl, vcl, cc...)",1.7%,1.4%,1.5%
có từ tục (VULGAR),0.6%,0.3%,0.4%
không có từ tục lẫn cảm thán,97.8%,98.4%,98.1%
ngắn (<= 5 từ),30.6%,31.2%,30.7%


In [15]:
# các token làm mô hình dễ bị báo động giả nhất
def tokens(s): 
    return set(re.findall(r"\w+", s.lower()))

p = VT[MODELS[-1]]
t, fl = toxic(p), flagged(p)
fp, ok = fl & ~t, ~fl & ~t

cf = Counter() # Đếm từ trong câu FP
co = Counter() # Đếm từ trong câu OK đoán đúng

for s in p.free_text[fp]: 
    cf.update(tokens(s))
for s in p.free_text[ok]: 
    co.update(tokens(s))

nf, no = fp.sum(), ok.sum()

# đo mức độ thiên vị nghiêng về FP của từng từ (chỉ xét từ xuất hiện >= 8 lần)
rows_token = []
for w, c in cf.items():
    if c >= 8:
        prob_fp = (c + 0.5) / (nf + 1)
        prob_ok = (co[w] + 0.5) / (no + 1)
        log_odds_val = np.log(prob_fp / prob_ok)
        
        rows_token.append({
            "token": w, 
            "n trong FP": c, 
            "n trong BT-đúng": co[w], 
            "log-odds": log_odds_val
        })

df_tokens = pd.DataFrame(rows_token)
display(
    df_tokens.sort_values("log-odds", ascending=False)
    .head(15)
    .round(2)
    .set_index("token")
    .rename_axis(f"token nghiêng về FP ({MODELS[-1]})")
)

,n trong FP,n trong BT-đúng,log-odds
token nghiêng về FP (visobert),,,
chó,9,2,3.95
đéo,15,5,3.65
no,8,15,2.02
kinh,9,19,1.90
mày,12,27,1.83
lam,10,23,1.81
ma,13,30,1.80
thằng,12,33,1.63
toàn,11,32,1.58


In [16]:
# lọc câu BÌNH THƯỜNG cả 3 mô hình cùng báo động giả (trên test)

fp_all = np.logical_and.reduce([flagged(P[m]) & ~toxic(P[m]) for m in MODELS])
print(f"{int(fp_all.sum())} câu BÌNH THƯỜNG bị cả {len(MODELS)} mô hình cùng báo động giả (test)")

q = P[MODELS[-1]]
view = q[fp_all].sort_values("confidence", ascending=False)[["free_text", "pred_label", "confidence"]].head(12)

display(view.assign(free_text=view.free_text.str.slice(0, 130)))

59 câu BÌNH THƯỜNG bị cả 3 mô hình cùng báo động giả (test)


,free_text,pred_label,confidence
1335,Dkm đừng chưởi tau nữa để tau còn ăn,XÚC PHẠM,0.9981
28,vl tên ạ =)),XÚC PHẠM,0.9974
1128,Ditme càng tốt =)) đỡ phải gánh tụt quần :>,XÚC PHẠM,0.9973
345,"ăn rồi cứu tú , cứu mà ăn lol à",XÚC PHẠM,0.9968
2453,Cạo lông lol há há há há 😂😂😂😂,XÚC PHẠM,0.9968
2056,"Kiên *chương. Mấy lỗi “tr”, “ch” nhìn khó chịu vl",XÚC PHẠM,0.9967
1250,Dkm tôi thề đỉnh vcl mấy ông ạ 😳😳,XÚC PHẠM,0.9963
2046,Sáng nào kao cũng thức lúc 4h sàng hết mài thằng assmin xăm xí đú...khỏi nhắc mài...hơ hơ hơ...,THÙ GHÉT,0.9954
691,Đéo bjt là mua hay là thu :)))),XÚC PHẠM,0.9953
1720,Mấy bạn dlv vào đính chính thông tin đi xem nào?,THÙ GHÉT,0.9947


In [17]:
# nhóm nguyên nhân báo động giả theo nhãn thủ công
b = VT[MODELS[0]]

fp_all_vt = np.logical_and.reduce([flagged(VT[m]) & ~toxic(VT[m]) for m in MODELS])

# Lọc các câu có trong tập đã gán nhãn thủ công
fa = b[fp_all_vt & b.id.isin(FP_ANNOT).values][["id", "free_text"]].copy()
fa["loại"] = fa.id.map(FP_ANNOT)

print(f"\nCả {len(MODELS)} mô hình cùng báo động giả: {int(fp_all_vt.sum())} câu (val+test); đã gán nhãn thủ công {len(fa)} câu để minh họa")

display(fa.groupby("loại").size().sort_values(ascending=False).rename("số câu"))

for k, g in fa.groupby("loại"):
    print(f"\n[{k}]")
    display(
        g[["id", "free_text"]]
        .assign(free_text=lambda d: d.free_text.str.slice(0, 140))
        .head(4)
    )


Cả 3 mô hình cùng báo động giả: 117 câu (val+test); đã gán nhãn thủ công 25 câu để minh họa


loại
tục dùng nhấn mạnh/đùa         10
nghi nhiễu nhãn                 7
chủ đề chính trị / nhà nước     5
không dấu gây nhầm từ           3
Name: số câu, dtype: int64


[chủ đề chính trị / nhà nước]


,id,free_text
1418,6086,Hèn gì ai cũng cố gắng bán nhà bán đất bán ruộng bán trâu để cho con em vào làm nhà nước
1858,1405,"Kể cả thế, bh báo chính thống đưa, thứ bộ trưởng phát biểu mới tin"
2634,26013,Đây là ác mộng của đảng cộng sản trung quốc nói riêng vs là ác mộng XHCN trên toàn thế giới nói chung 😂
2691,10614,"Quí vị cũng đừng chửi rũa ông này làm jì, chẳng qua ông này chỉ làm theo lệnh cấp trên mà thôi,nói chung trong chế độ xhcn tập quyền của tru"



[không dấu gây nhầm từ]


,id,free_text
365,27439,Đừng cố biện minh =)))) choi lon
2845,22495,NO banh xac
5701,33773,Con nay bo tay



[nghi nhiễu nhãn]


,id,free_text
755,25366,tao méc mấy a dlv tụi bây phản động nè
916,21350,"Cái đụ bà các thánh làm em cười đéo nhặt được buồi, nhầm mồm. Đặc sản VN đây r 😁"
2263,25287,hoàn cảnh cả thôi ! đường đời chúng mày gặp trông gai nếu không nhổ vứt ra đường thì chẳng lẽ đem về cắm vào đít thằng khác à
3166,22622,"Bon nước ngoài này không thể tin thằng nào được,"



[tục dùng nhấn mạnh/đùa]


,id,free_text
249,12655,""" tôi xem phim lại đéo biết khóc bao giờ "" =))"
292,13910,vcl :))))
786,4205,dỗi vl :v
1628,2657,Đéo phải ông già mình đâu các bạn


**Nhận xét FP đối với câu trung tính**

Có khoảng 76 - 79% báo động giả không chứa từ tục, cảm thán viết tắt trong từ điển (so với 98% ở câu BÌNH THƯỜNG đoán đúng). mô hình "bắt từ khóa" và dựa vào dấu hiệu gián tiếp: xưng hô (mày, thằng, nó), từ "chó", "đéo", tiếng không dấu (token `no`, `lam`, `ma`, `co` đứng đầu danh sách), chủ đề chính trị

117 câu cả 3 mô hình cùng báo động giả và phần đã gán nhãn thủ công, có 4 nhóm: 
1. tục dùng nhấn mạnh/đùa ("vcl", "đéo", "dỗi vl")
2. chính trị/nhà nước nhưng không công kích
3. không dấu gây nhầm từ ("choi lon", "Con nay bo tay")
4. nghi nhiễu nhãn: câu rõ ràng công kích ("Bon nước ngoài này không thể tin thằng nào được", "đường đời chúng mày gặp...cắm vào đít thằng khác") nhưng nhãn gốc là BÌNH THƯỜNG. Nhóm (4) nghĩa là một phần "báo động giả" thực ra là **nhãn sai**, nên FPR thật của mô hình có thể thấp hơn số đo

## 6. False negative với ẩn dụ, nói lái và nói vòng

In [18]:
ANNOT = {
    # mỉa mai / ẩn dụ - không có từ tục, nghĩa nằm ở ngữ cảnh
    21949: "mỉa mai / ẩn dụ", 2276: "mỉa mai / ẩn dụ", 13911: "mỉa mai / ẩn dụ", 28307: "mỉa mai / ẩn dụ", 414: "mỉa mai / ẩn dụ", 23098: "mỉa mai / ẩn dụ",
    5327: "mỉa mai / ẩn dụ", 3696: "mỉa mai / ẩn dụ", 1454: "mỉa mai / ẩn dụ", 5501: "mỉa mai / ẩn dụ",
    # đe dọa / nguyền rủa nói vòng
    13443: "đe dọa / nguyền rủa nói vòng", 1202: "đe dọa / nguyền rủa nói vòng", 5861: "đe dọa / nguyền rủa nói vòng", 1018: "đe dọa / nguyền rủa nói vòng",
    # tên gọi lóng miệt thị nhóm (quốc tịch, vùng miền...)
    10410: "tên gọi lóng miệt thị nhóm", 1168: "tên gọi lóng miệt thị nhóm", 15890: "tên gọi lóng miệt thị nhóm", 28217: "tên gọi lóng miệt thị nhóm", 7999: "tên gọi lóng miệt thị nhóm",
    # biến thể chính tả: bỏ dấu, viết dính, biến âm, chèn ký tự (tục nhưng không khớp từ điển)
    6804: "biến thể chính tả / che chữ", 17657: "biến thể chính tả / che chữ", 4012: "biến thể chính tả / che chữ", 1237: "biến thể chính tả / che chữ",
    7176: "biến thể chính tả / che chữ", 20760: "biến thể chính tả / che chữ", 24805: "biến thể chính tả / che chữ", 12910: "biến thể chính tả / che chữ",
    9540: "biến thể chính tả / che chữ", 23650: "biến thể chính tả / che chữ",
    # nội dung tình dục ẩn ý
    11371: "ẩn ý tình dục", 3783: "ẩn ý tình dục",
    # quá ngắn / thiếu ngữ cảnh / nhãn khó đoán
    6848: "quá ngắn / thiếu ngữ cảnh", 25714: "quá ngắn / thiếu ngữ cảnh", 27074: "quá ngắn / thiếu ngữ cảnh", 33813: "quá ngắn / thiếu ngữ cảnh",
}

In [19]:
# thống kê tỷ lệ bỏ sót có chứa từ tục hay kh
rows = []
for m in MODELS:
    p = VT[m] 
    t, fl = toxic(p), flagged(p)
    fn = t & ~fl
    explicit = has_term(p.free_text, VULGAR_RE) | has_term(p.free_text, EXPLETIVE_RE)
    
    fn_explicit = int((fn & explicit).sum())
    fn_implicit = int((fn & ~explicit).sum())
    total_fn = int(fn.sum())
    
    total_explicit = int((t & explicit).sum())
    total_implicit = int((t & ~explicit).sum())
    
    rows.append(
        {
           "mô hình": m, 
            "FN": total_fn, 
            "FN có từ tục": fn_explicit, 
            "FN không có từ tục (ẩn ý/nói vòng)": fn_implicit,
            "% FN không có từ tục": pct(fn_implicit, total_fn),
            "FNR | có từ tục": pct(fn_explicit, total_explicit), 
            "FNR | không có từ tục": pct(fn_implicit, total_implicit)
        }
    )
display(pd.DataFrame(rows).set_index("mô hình"))

# phân tích câu độc hại bị cả 3 mô hình cùng bỏ sót dựa trên nhãn thủ công
b = VT[MODELS[0]]

fn_all = np.logical_and.reduce([toxic(VT[m]) & ~flagged(VT[m]) for m in MODELS])

ann = b[fn_all & b.id.isin(ANNOT).values][["id", "free_text", "true_label"]].copy()
ann["loại"] = ann.id.map(ANNOT)

print(f"Cả {len(MODELS)} mô hình cùng bỏ sót: {int(fn_all.sum())} câu (val+test); đã gán nhãn thủ công {len(ann)} câu để minh họa")

display(ann.groupby("loại").size().sort_values(ascending=False).rename("số câu"))

for k, g in ann.groupby("loại"):
    print(f"\n[{k}]")
    display(
        g[["id", "free_text", "true_label"]]
        .assign(free_text=lambda d: d.free_text.str.slice(0, 140))
        .head(5)
    )

,FN,FN có từ tục,FN không có từ tục (ẩn ý/nói vòng),% FN không có từ tục,FNR | có từ tục,FNR | không có từ tục
mô hình,,,,,,
baseline,317,50,267,84.2%,11.5%,42.9%
phobert,219,22,197,90.0%,5.1%,31.6%
visobert,291,43,248,85.2%,9.9%,39.8%


Cả 3 mô hình cùng bỏ sót: 123 câu (val+test); đã gán nhãn thủ công 35 câu để minh họa


loại
biến thể chính tả / che chữ     10
mỉa mai / ẩn dụ                 10
tên gọi lóng miệt thị nhóm       5
quá ngắn / thiếu ngữ cảnh        4
đe dọa / nguyền rủa nói vòng     4
ẩn ý tình dục                    2
Name: số câu, dtype: int64


[biến thể chính tả / che chữ]


,id,free_text,true_label
58,20760,Khau nghiep,XÚC PHẠM
817,7176,@Vh Fg vo van hoa vo hoc,THÙ GHÉT
973,12910,"TIN VUI ,ĐANG CÔNG SAN VN BI NHIÊM VIRUS CORONA CHÊT GÂN HÊT RÔI HAHAHA😂😅😃😆👍👍👏👏",THÙ GHÉT
2411,23650,:>>>>>> taocutaodiia :)))) nghiệp quá cô 3 ư :>>,THÙ GHÉT
2708,6804,Chúc mừg abe căc hơi to nha !,THÙ GHÉT


...

,id,free_text,true_label
1406,15890,Dân InDo m chọc mù,THÙ GHÉT
2133,1168,khựa nhé :)))),XÚC PHẠM
3693,28217,"Hà Nguyễn Khánh Duy Gái Nhật đi tàu đi ngầm nó ngứa mu quá, lấy tay móc gãi tỉnh queo. Nhìn ghê lắm kìa",THÙ GHÉT
5451,10410,"Ở indonesia đã có tàu điện ngầm rồi, ở Đông Lào đợi mãi không có",THÙ GHÉT
5668,7999,Thiếu nốt cái live là anh chăng khác gì bọn cô rê an,THÙ GHÉT



[đe dọa / nguyền rủa nói vòng]


,id,free_text,true_label
2430,1018,bắn bỏ,XÚC PHẠM
3571,5861,Thờ buổi này đi du lịch chi vậy 😀 đi để kịp đầu thai hả ha gì mà đi dữ vậy,THÙ GHÉT
4370,1202,"giác ngộ đi , tp HCM nhé , bác giết SG rồi 😆😆😆😆",THÙ GHÉT
4891,13443,Vậy là anh muốn vô quan tài ở hả,THÙ GHÉT



[ẩn ý tình dục]


,id,free_text,true_label
2641,3783,Hạnh fuck nhé đôi bạn trẻ 🤣,XÚC PHẠM
3169,11371,Sờ chim a ơu,XÚC PHẠM


**Nhận xét FN câu ẩn dụ, nói lái**
 
84-90% số FN không chứa từ tục trong từ điển. FNR của câu độc hại có từ tục chỉ 5 - 12%, còn câu không có từ tục là 32 - 43%. Bỏ sót chủ yếu là lỗi thiếu ngữ cảnh/hiểu ngầm, không phải lỗi bắt từ khóa

Các nhóm bỏ sót khi rà thủ công (35 câu gán nhãn) mà nhãn gốc khó đoán: 
- mỉa mai/ẩn dụ chính trị-xã hội ("câu chuyện cổ tích ... công an", "Tưởng Nhật Bản văn minh thế nào")
- đe dọa/nguyền rủa nói vòng ("muốn vô quan tài ở hả", "kịp đầu thai hả")
- tên gọi lóng miệt thị nhóm ("khựa", "Đông Lào", "cô rê an")
- biến thể chính tả/che chữ không dấu hoặc viết dính ("vo van hoa vo hoc", "xaolon", "l.o.n", "căc")
- ẩn ý tình dục

**Nói lái đúng nghĩa (đảo âm/vần) gần như không gặp** trong các câu đã rà. gặp nhiều hơn là biến âm, bỏ dấu... ("lòn", "căc", "del", "thèn")